# Step 5 — Fine-tune on drawn seeds (for the interactive tool)

The FNO was trained on `hex`/`point`/`multi` seeds. The Crystal Studio UI lets a
user draw **arbitrary** shapes, which are out of distribution. This notebook:

1. generates a **drawn-seed dataset** (random blobs + strokes, per-region random
   orientations) with the exact PFC solver,
2. **warm-starts** from an existing grain model and fine-tunes on
   grain + drawn data together (so it learns drawn shapes without forgetting),
3. measures the payoff: **rollout MSE on a held-out drawn test set, before vs
   after** fine-tuning.

The solver runs only for data generation; fine-tuning is the usual FNO training.
Set **Runtime → T4 GPU**, then **Run all**.

### 1. Mount Drive + clone + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Tntguy0128/crystal-growth-sim.git
%cd crystal-growth-sim
!pip install -q torch numpy scipy pyyaml matplotlib tensorboard
import sys; sys.path.insert(0, 'crystal_tool')      # tool modules (generator, analyze)

### 2. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then Run all."
print("GPU OK:", torch.cuda.get_device_name(0))

### 3. Grain-rich dataset (reuse from Drive or generate)

In [ ]:
import os
z = '/content/drive/MyDrive/pfc_dataset_grains.zip'
if os.path.exists(z):
    !cp "$z" .; unzip -o -q pfc_dataset_grains.zip
else:
    !python generate_dataset.py --sweep --config pfc_config_grains.yaml --parallel --max-workers 2
    !zip -r -q pfc_dataset_grains.zip data_pfc_grains; cp pfc_dataset_grains.zip /content/drive/MyDrive/
print('grain trajectories:', len(os.listdir('data_pfc_grains')))

### 4. Drawn datasets — training set + a DISJOINT held-out test set
The test set uses different RNG seeds and run ids, so it shares no seed or file
with the training set: a clean before/after benchmark.

In [ ]:
import os
# (dir, n_traj, rng_base, id_base)
for d, n, rb, ib in [('data_pfc_drawn', 100, 5000, 1000),
                     ('data_pfc_drawn_test', 20, 9000, 2000)]:
    z = f'/content/drive/MyDrive/{d}.zip'
    if os.path.exists(z):
        !cp "$z" .; unzip -o -q {d}.zip
    else:
        !python crystal_tool/generate_drawn_dataset.py --n {n} --output-dir {d} --rng-base {rb} --id-base {ib}
        !zip -r -q {d}.zip {d}; cp {d}.zip /content/drive/MyDrive/
    print(d, '->', len(os.listdir(d)), 'files')

# quick quality check on a few drawn finals
import numpy as np, glob, analyze as A
for p in sorted(glob.glob('data_pfc_drawn/*.npz'))[:4]:
    zz = np.load(p, allow_pickle=True)
    pr = A.analyze(zz['n_all'][-1], float(zz['dx']))
    print(f"  {os.path.basename(p)}: cryst={pr['crystallinity']:.2f} "
          f"grains={pr['n_grains']} defects={pr['n_defects']} "
          f"orient={pr['dominant_orientation_deg']:.0f}deg")

### 5. Combined fine-tune set = grain + drawn
The held-out `data_pfc_drawn_test` is kept OUT of this set. Drawn files carry
`seed_type=custom_mask` in their npz, so `split_by: config` groups them cleanly
(no manifest needed; grain files keep their manifest).

In [ ]:
import os, glob, shutil
os.makedirs('data_pfc_finetune', exist_ok=True)
for f in glob.glob('data_pfc_grains/*.npz') + glob.glob('data_pfc_drawn/*.npz'):
    shutil.copy(f, 'data_pfc_finetune/')
if os.path.exists('data_pfc_grains/manifest.csv'):
    shutil.copy('data_pfc_grains/manifest.csv', 'data_pfc_finetune/')
print('fine-tune trajectories:', len(glob.glob('data_pfc_finetune/*.npz')))

### 6. Warm-start fine-tune
`BASE` is the checkpoint to fine-tune FROM — point it at one of your grain models
on Drive. Low LR + few epochs so it adapts to drawn seeds without overwriting
what it already knows.

In [ ]:
import yaml, subprocess, shutil, os
BASE = '/content/drive/MyDrive/fno_grains_nophysics.pt'   # <-- your base checkpoint
assert os.path.exists(BASE), f"missing base checkpoint: {BASE} (change BASE)"

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['data']['data_dir'] = 'data_pfc_finetune'
cfg['data']['split_by'] = 'config'
cfg['train']['init_checkpoint'] = BASE
cfg['train']['lr'] = 1e-4            # gentle: fine-tune, don't retrain
cfg['train']['epochs'] = 40
cfg['train']['patience'] = 12
cfg['train']['physics_weight'] = 0.0
cfg['train']['pde_weight'] = 0.0
out_dir = 'runs/finetune_drawn'
cfg['logging']['out_dir'] = out_dir
cfg['eval']['out_dir'] = f'{out_dir}/eval'
with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
print('===== FINE-TUNE (warm-start from base) =====', flush=True)
subprocess.run(['python', '-u', 'train_fno.py', '--config', 'config.yaml'], check=True, env=env)
shutil.copy(f'{out_dir}/best.pt', '/content/drive/MyDrive/fno_finetune_drawn.pt')
print('saved -> Drive/fno_finetune_drawn.pt', flush=True)

### 7. The payoff — drawn-seed rollout MSE, before vs after
Same held-out drawn test set for both models. Lower is better; the fine-tune
should cut the error on drawn seeds.

In [ ]:
import torch, numpy as np, glob
from fno_model import build_model
from dataset import inspect_trajectory

def _load(ckpt):
    ck = torch.load(ckpt, map_location='cuda', weights_only=False)
    cfg = ck['config']; cfg['model']['in_channels'] = ck.get('in_channels', 1)
    m = build_model(cfg).cuda(); m.load_state_dict(ck['model_state']); m.eval()
    return m, float(ck['norm_mean']), float(ck['norm_std'])

@torch.no_grad()
def rollout_mse(ckpt, files):
    m, mean, std = _load(ckpt); errs = []
    for f in files:
        rec = inspect_trajectory(f)
        fr = np.concatenate([rec['inputs'], rec['targets'][-1:]], 0).astype('float32')
        cur = torch.from_numpy((fr[0]-mean)/(std+1e-8)).float().view(1,1,*fr[0].shape).cuda()
        preds = [fr[0]]
        for _ in range(len(fr)-1):
            cur = m(cur); preds.append(cur.cpu().numpy()[0,0]*(std+1e-8)+mean)
        preds = np.stack(preds)
        errs.append(float(np.mean((preds[1:]-fr[1:])**2)))
    return float(np.mean(errs))

test_files = sorted(glob.glob('data_pfc_drawn_test/*.npz'))
base_mse = rollout_mse(BASE, test_files)
ft_mse   = rollout_mse(f'{out_dir}/best.pt', test_files)
print(f'held-out drawn seeds ({len(test_files)} trajectories):')
print(f'  base       rollout MSE : {base_mse:.4e}')
print(f'  fine-tuned rollout MSE : {ft_mse:.4e}')
print(f'  improvement            : {100*(base_mse-ft_mse)/base_mse:+.1f}%')

### 8. Save eval to Drive

In [ ]:
import subprocess, os
env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
subprocess.run(['python','-u','evaluate_fno.py','--config','config.yaml',
                '--checkpoint', f'{out_dir}/best.pt'], check=True, env=env)
!zip -r -q fno_finetune_drawn_eval.zip runs/finetune_drawn/eval
!cp fno_finetune_drawn_eval.zip /content/drive/MyDrive/
print('saved: fno_finetune_drawn.pt + fno_finetune_drawn_eval.zip')
print('Point Crystal Studio at fno_finetune_drawn.pt for drawn-seed predictions.')